---
Accession Extraction Validation - Experiment Component

This module handles model inference on XML articles to extract accession numbers.
It supports multiple LLM providers with a unified interface.

Project: AI6129 Pathogen Tracking System
Version: 1.2 (Merged)

Model Configuration (Updated January 2026):
- Development: Claude Haiku 4.5
- Formal Experiments: Claude Sonnet 4 and Amazon Nova Pro via AWS Bedrock

Changes in v1.2:
- Native DSPy token tracking via lm.history
- Capture and log CoT reasoning for each document
- Store full conversation history in separate log file
- Configurable console output for reasoning
- Replaced request-based rate limiting with token-based rate limiting
- Increased retry delays for rate limit errors (30s minimum)
- Increased max_tokens to 8192 for large accession lists
- Added rate limit specific error handling in retry logic
- Updated DSPy signature for precision-focused extraction
- Updated to Claude Haiku 4.5
---

In [1]:
!pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.9 MB/s eta 0:00:00


In [2]:
import json
import os
import re
import time
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional, Any, Callable
from google.colab import drive
from google.colab import userdata
import xml.etree.ElementTree as ET

import dspy

In [3]:
# =============================================================================
# CONFIGURATION CONSTANTS
# =============================================================================

# DSPy model strings - validated for DSPy 3.x with LiteLLM
DSPY_MODEL_STRINGS = {
    # Baseline (Anthropic API - direct call, cost-effective)
    "claude-haiku-4.5": "anthropic/claude-haiku-4-5-20251001",  # changed - updated to Haiku 4.5

    # Formal Experiments (AWS Bedrock Singapore/APAC)
    "claude-sonnet-4": "bedrock/apac.anthropic.claude-sonnet-4-20250514-v1:0",
    "amazon-nova-pro": "bedrock/apac.amazon.nova-pro-v1:0",
}

# Alternative: Direct Singapore region model IDs (if APAC profile unavailable)
BEDROCK_SINGAPORE_DIRECT = {
    "amazon-nova-pro": "bedrock/amazon.nova-pro-v1:0",
    "amazon-nova-lite": "bedrock/amazon.nova-lite-v1:0",
    "amazon-nova-micro": "bedrock/amazon.nova-micro-v1:0",
}

# Default path for accession format rules file
DEFAULT_FORMAT_RULES_FILE = "accession_formats.md"

# Model pricing per 1M tokens (USD) - for cost calculation
MODEL_PRICING = {
    "claude-haiku-4.5": {"input": 1.00, "output": 5.00},  # changed - updated pricing
    "claude-sonnet-4": {"input": 3.00, "output": 15.00},
    "amazon-nova-pro": {"input": 0.80, "output": 3.20},
}

# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

# Set to True to print CoT reasoning to console during processing
PRINT_REASONING = False

# Set to True to print conversation history to console (very verbose)
PRINT_HISTORY = False

# Maximum characters of reasoning to print to console (if enabled)
MAX_REASONING_PRINT_LENGTH = 500

# =============================================================================
# RATE LIMIT CONFIGURATION  # changed - new section
# =============================================================================

# Rate limit configuration (Anthropic free tier: 50k tokens/min)
DEFAULT_TOKENS_PER_MINUTE = 40000  # changed - 40k leaves buffer below 50k limit

 ---
 TYPE DEFINITIONS
 ---

In [4]:
class ModelProvider(Enum):
    """Supported LLM providers"""
    ANTHROPIC_API = "anthropic_api"
    GEMINI_API = "gemini_api"
    GOOGLE_COLAB = "google_colab"
    AWS_BEDROCK = "aws_bedrock"


class ExperimentStatus(Enum):
    """Status of document processing"""
    COMPLETED = "completed"
    FAILED = "failed"
    SKIPPED = "skipped"


@dataclass
class ModelConfig:
    """
    Configuration for a single model.
    """
    model_id: str
    provider: ModelProvider
    dspy_model_string: str
    max_tokens: int = 8192  # changed - increased from 4096 for large accession lists
    temperature: float = 0.0
    tokens_per_minute: int = 40000  # changed - token-based rate limit
    region: Optional[str] = None


@dataclass
class ArticleContent:
    """Processed article ready for LLM input"""
    pmcid: str
    xml_filepath: str
    raw_xml: str
    extracted_text: str
    sections: Dict[str, str]
    tables: List[str]
    token_count_estimate: int


@dataclass
class ExtractionResult:
    """Result from processing a single document"""
    pmcid: str
    extracted_accessions: List[str]
    raw_model_output: str
    processing_time_ms: int
    token_count_input: int
    token_count_output: int
    tokens_are_estimated: bool  # changed - added field
    cost_usd: float
    model_id: str
    timestamp: str
    status: ExperimentStatus
    error_message: Optional[str]


@dataclass
class ExperimentCheckpoint:
    """Checkpoint for resuming interrupted experiments"""
    experiment_id: str
    model_id: str
    split_name: str
    completed_pmcids: Set[str]
    failed_pmcids: Dict[str, str]
    last_updated: str
    total_documents: int


@dataclass
class ExperimentMetadata:
    """Metadata about an experiment run"""
    experiment_id: str
    model_id: str
    dspy_version: str
    run_timestamp: str
    split_name: str
    total_documents: int
    completed_documents: int
    failed_documents: int
    total_processing_time_ms: int
    total_tokens_input: int
    total_tokens_output: int
    total_cost_usd: float


@dataclass
class ExperimentOutput:
    """Final output structure consumed by evaluation component"""
    experiment_metadata: ExperimentMetadata
    predictions: List[ExtractionResult]


@dataclass
class TokenRateLimiter:  # changed - renamed from RateLimiter
    """Token-based sliding window rate limiter for API calls"""
    tokens_per_minute: int
    token_history: List[Tuple[float, int]] = field(default_factory=list)  # (timestamp, token_count)

---
 MODEL CONFIGURATION
---

In [5]:
def create_model_configs() -> Dict[str, ModelConfig]:
    """
    Create configuration for all models to be tested.

    Returns:
        Mapping of model_id -> ModelConfig
    """
    configs = {}

    # Baseline: Claude Haiku 4.5 via Anthropic API  # changed
    configs["claude-haiku-4.5"] = ModelConfig(  # changed
        model_id="claude-haiku-4.5",  # changed
        provider=ModelProvider.ANTHROPIC_API,
        dspy_model_string=DSPY_MODEL_STRINGS["claude-haiku-4.5"],  # changed
        max_tokens=8192,  # changed - increased from 4096
        temperature=0.0,
        tokens_per_minute=40000,  # changed - token-based limit
        region=None
    )

    # Comparison A: Claude Sonnet 4 via AWS Bedrock APAC
    configs["claude-sonnet-4"] = ModelConfig(
        model_id="claude-sonnet-4",
        provider=ModelProvider.AWS_BEDROCK,
        dspy_model_string=DSPY_MODEL_STRINGS["claude-sonnet-4"],
        max_tokens=8192,  # changed
        temperature=0.0,
        tokens_per_minute=80000,  # changed - Bedrock may have higher limits
        region="ap-southeast-1"
    )

    # Comparison B: Amazon Nova Pro via AWS Bedrock APAC
    configs["amazon-nova-pro"] = ModelConfig(
        model_id="amazon-nova-pro",
        provider=ModelProvider.AWS_BEDROCK,
        dspy_model_string=DSPY_MODEL_STRINGS["amazon-nova-pro"],
        max_tokens=8192,  # changed
        temperature=0.0,
        tokens_per_minute=80000,  # changed
        region="ap-southeast-1"
    )

    return configs


def initialise_dspy_model(config: ModelConfig) -> dspy.LM:
    """
    Initialise DSPy language model based on provider configuration.

    Parameters:
        config: Model configuration

    Returns:
        Configured DSPy LM object
    """
    lm = None

    if config.provider == ModelProvider.ANTHROPIC_API:
        lm = dspy.LM(
            config.dspy_model_string,
            temperature=config.temperature,
            max_tokens=config.max_tokens
        )
    elif config.provider == ModelProvider.GEMINI_API:
        lm = dspy.LM(
            config.dspy_model_string,
            temperature=config.temperature,
            max_tokens=config.max_tokens
        )
    elif config.provider == ModelProvider.AWS_BEDROCK:
        os.environ["AWS_REGION_NAME"] = config.region or "ap-southeast-1"

        lm = dspy.LM(
            config.dspy_model_string,
            temperature=config.temperature,
            max_tokens=config.max_tokens
        )

    return lm

---
 FORMAT RULES LOADING
---

In [6]:
def load_format_rules(filepath: str) -> str:
    """
    Load accession number format rules from markdown file.
    """
    format_path = Path(filepath)

    if not format_path.exists():
        raise FileNotFoundError(f"Format rules file not found: {filepath}")

    try:
        with open(format_path, 'r', encoding='utf-8') as f:
            content = f.read()

        print(f"Format rules loaded from: {filepath} ({len(content)} characters)")
        return content

    except Exception as e:
        raise IOError(f"Error reading format rule file: {e}")

---
XML PROCESSING FUNCTIONS
---

In [7]:
def load_xml_articles(xml_filepath: str) -> str:
    """Load raw XML content from file."""
    with open(xml_filepath, 'r', encoding='utf-8') as f:
        return f.read()


def extract_text_content(element: ET.Element) -> str:
    """Extract all text content from an XML element."""
    texts = []

    if element.text:
        texts.append(element.text.strip())

    for child in element:
        texts.append(extract_text_content(child))
        if child.tail:
            texts.append(child.tail.strip())

    return " ".join(t for t in texts if t)


def extract_article_sections(raw_xml: str) -> Dict[str, str]:
    """
    Parse XML and extract content by section.
    """
    sections = {}

    try:
        root = ET.fromstring(raw_xml)
    except ET.ParseError:
        return sections

    # Extract abstract
    abstract_elem = root.find(".//abstract")
    if abstract_elem is not None:
        sections["abstract"] = extract_text_content(abstract_elem)

    # Extract body sections
    body_elem = root.find(".//body")
    if body_elem is not None:
        section_elem = body_elem.findall(".//sec")

        for section in section_elem:
            title_elem = section.find(".//title")
            section_title = title_elem.text if title_elem is not None and title_elem.text else ""
            section_content = extract_text_content(section)

            if section_title:
                sections[section_title.lower().strip()] = section_content
            else:
                sections[f"section_{len(sections)}"] = section_content

    # Extract back matter (data availability, acknowledgement)
    back_elem = root.find(".//back")
    if back_elem is not None:
        sections["back_matter"] = extract_text_content(back_elem)

    return sections


def extract_tables_from_xml(raw_xml: str) -> List[str]:
    """
    Extract table content from XML.
    """
    tables = []

    try:
        root = ET.fromstring(raw_xml)
    except ET.ParseError:
        return tables

    table_wraps = root.findall(".//table-wrap")

    for table_wrap in table_wraps:
        label_elem = table_wrap.find(".//label")
        caption_elem = table_wrap.find(".//caption")

        label = label_elem.text if label_elem is not None and label_elem.text else ""
        caption = extract_text_content(caption_elem) if caption_elem is not None else ""

        table_content = convert_table_to_text(table_wrap)
        formatted_table = f"{label}: {caption}\n{table_content}"
        tables.append(formatted_table)

    return tables


def convert_table_to_text(table_element: ET.Element) -> str:
    """
    Convert XML table element to readable text format.
    """
    rows = []

    # Process header
    thead = table_element.find(".//thead")
    if thead is not None:
        header_row = thead.find(".//tr")
        if header_row is not None:
            cells = [extract_text_content(cell) for cell in header_row.findall(".//th")]
            rows.append("|".join(cells))
            rows.append("-" * 40)

    # Process body rows
    tbody = table_element.find(".//tbody")
    if tbody is not None:
        body_rows = tbody.findall(".//tr")
        for row in body_rows:
            cells = [extract_text_content(cell) for cell in row.findall(".//td")]
            rows.append("|".join(cells))

    return "\n".join(rows)


def estimate_token_count(text: str) -> int:
    """Estimate token count (approximately 4 characters per token)."""
    return len(text) // 4


def prepare_article_content(
    pmcid: str,
    xml_filepath: str,
    max_tokens: int = 100000
) -> ArticleContent:
    """
    Load and process XML article for LLM input.
    """
    raw_xml = load_xml_articles(xml_filepath)
    sections = extract_article_sections(raw_xml)
    tables = extract_tables_from_xml(raw_xml)

    extracted_text = compose_llm_input(sections, tables)
    token_estimate = estimate_token_count(extracted_text)

    if token_estimate > max_tokens:
        extracted_text = truncate_text_smart(extracted_text, max_tokens)
        token_estimate = max_tokens

    return ArticleContent(
        pmcid=pmcid,
        xml_filepath=xml_filepath,
        raw_xml=raw_xml,
        extracted_text=extracted_text,
        sections=sections,
        tables=tables,
        token_count_estimate=token_estimate
    )


def compose_llm_input(sections: Dict[str, str], tables: List[str]) -> str:
    """
    Compose sections and tables into LLM input text.
    """
    parts = []

    for section_name, section_content in sections.items():
        parts.append(f"[SECTION: {section_name.upper()}]")
        parts.append(section_content)
        parts.append("")

    if tables:
        parts.append("[TABLES]")
        for idx, table_content in enumerate(tables):
            parts.append(f"[TABLE {idx + 1}]")
            parts.append(table_content)
            parts.append("")

    return "\n".join(parts)


def truncate_text_smart(text: str, max_tokens: int) -> str:
    """
    Truncate text preserving beginning and end.
    """
    max_chars = max_tokens * 4

    if len(text) <= max_chars:
        return text

    front_chars = int(max_chars * 0.7)
    back_chars = max_chars - front_chars - 50

    front_text = text[:front_chars]
    back_text = text[-back_chars:]

    return f"{front_text}\n\n[.... CONTENT TRUNCATED .....]\n\n{back_text}"

---
DSPY SIGNATURE AND MODULE
---

In [8]:
# changed - replaced entire signature with precision-focused version from Copy_of
class AccessionExtractionSignature(dspy.Signature):
    """Extract accession identifiers that appear verbatim in the article."""

    format_rules: str = dspy.InputField(
        desc=(
            "Markdown reference for valid accession formats and known false positives. "
            "Use it ONLY to validate candidates; do not copy examples from it."
        )
    )

    xml_content: str = dspy.InputField(
        desc="Full article XML to extract from."
    )

    accessions: list[str] = dspy.OutputField(
        desc=(
            "Unique accession strings found in xml_content (verbatim). "
            "Return [] if none. Do not include uncertain candidates."
        )
    )

    notes: list[str] = dspy.OutputField(
        desc="Optional brief notes about exclusions/uncertainty; keep short."
    )


class AccessionExtractor(dspy.Module):
    """
    DSPy module for accession extraction with chain-of-thought reasoning.
    """
    def __init__(self):
        super().__init__()
        self.predictor = dspy.ChainOfThought(AccessionExtractionSignature)

    def forward(self, format_rules: str, xml_content: str) -> dspy.Prediction:  # changed - parameter name
        """
        Extract accession numbers from article text using format rules as context.
        """
        prediction = self.predictor(
            format_rules=format_rules,
            xml_content=xml_content  # changed - parameter name
        )
        return prediction


def create_accession_extractor() -> AccessionExtractor:
    """Factory function to create the extractor module."""
    return AccessionExtractor()

---
TOKEN-BASED RATE LIMITING
---

In [9]:
# changed - entire section replaced with token-based rate limiting
def create_token_rate_limiter(tokens_per_minute: int = DEFAULT_TOKENS_PER_MINUTE) -> TokenRateLimiter:
    """
    Create token-based rate limiter.

    Parameters:
        tokens_per_minute: Maximum tokens allowed per minute (default 40k, below 50k API limit)

    Returns:
        TokenRateLimiter instance
    """
    return TokenRateLimiter(tokens_per_minute=tokens_per_minute, token_history=[])


def get_tokens_used_in_window(rate_limiter: TokenRateLimiter) -> int:
    """
    Get total tokens used in the current 60-second window.

    Returns:
        Number of tokens used in the last 60 seconds
    """
    current_time = time.time()
    window_start = current_time - 60.0

    # Clean up old entries
    rate_limiter.token_history = [
        (ts, tokens) for ts, tokens in rate_limiter.token_history
        if ts > window_start
    ]

    return sum(tokens for _, tokens in rate_limiter.token_history)


def wait_for_token_rate_limit(rate_limiter: TokenRateLimiter, estimated_tokens: int) -> None:
    """
    Block execution if token rate limit would be exceeded.

    This function implements a sliding window rate limiter based on tokens
    rather than request count, which is more appropriate for LLM APIs.

    Parameters:
        rate_limiter: TokenRateLimiter instance
        estimated_tokens: Estimated input tokens for the next request
    """
    current_time = time.time()
    window_start = current_time - 60.0

    # Remove entries outside the 60-second window
    rate_limiter.token_history = [
        (ts, tokens) for ts, tokens in rate_limiter.token_history
        if ts > window_start
    ]

    # Calculate tokens used in current window
    tokens_in_window = sum(tokens for _, tokens in rate_limiter.token_history)

    # Check if adding this request would exceed limit
    if tokens_in_window + estimated_tokens > rate_limiter.tokens_per_minute:
        # Calculate wait time until enough tokens are freed
        if rate_limiter.token_history:
            # Find how long until we have enough capacity
            tokens_needed = (tokens_in_window + estimated_tokens) - rate_limiter.tokens_per_minute
            tokens_freed = 0
            wait_until = current_time

            # Sort by timestamp to process oldest first
            sorted_history = sorted(rate_limiter.token_history, key=lambda x: x[0])

            for ts, tokens in sorted_history:
                tokens_freed += tokens
                wait_until = ts + 60.0
                if tokens_freed >= tokens_needed:
                    break

            wait_time = wait_until - current_time + 1.0  # Add 1s buffer

            if wait_time > 0:
                print(f"Token rate limit: {tokens_in_window:,}/{rate_limiter.tokens_per_minute:,} used. "
                      f"Need {estimated_tokens:,} tokens. Waiting {wait_time:.1f}s...")
                time.sleep(wait_time)

                # Clean up history after waiting
                current_time = time.time()
                window_start = current_time - 60.0
                rate_limiter.token_history = [
                    (ts, tokens) for ts, tokens in rate_limiter.token_history
                    if ts > window_start
                ]


def record_token_usage(rate_limiter: TokenRateLimiter, actual_tokens: int) -> None:
    """
    Record actual token usage after API call completes.

    Parameters:
        rate_limiter: TokenRateLimiter instance
        actual_tokens: Actual input tokens used
    """
    rate_limiter.token_history.append((time.time(), actual_tokens))

---
RETRY LOGIC WITH RATE LIMIT HANDLING
---

In [10]:
# changed - enhanced retry logic with rate limit detection
def execute_with_retry(func: Callable,
                       max_retries: int = 5,  # changed - increased from 3
                       base_delay: float = 5.0,  # changed - increased from 1.0
                       max_delay: float = 120.0  # changed - increased from 60.0
                       ) -> Any:
    """
    Execute function with exponential backoff retry.

    For rate limit errors, uses longer delays (minimum 30s) since the
    Anthropic API rate limit window is 60 seconds.

    Parameters:
        func: Function to execute
        max_retries: Maximum retry attempts (default 5)
        base_delay: Initial delay between retries in seconds (default 5.0)
        max_delay: Maximum delay cap in seconds (default 120.0)

    Returns:
        Function result if successful

    Raises:
        Exception: If all retries exhausted
    """
    delay = base_delay
    last_exception = None

    for attempt in range(max_retries):
        try:
            result = func()
            return result

        except Exception as e:
            last_exception = e
            error_name = type(e).__name__
            error_str = str(e).lower()

            # Check if this is a rate limit error  # changed
            is_rate_limit = ('rate_limit' in error_str or
                            'rate limit' in error_str or
                            'ratelimit' in error_str or
                            '429' in error_str)

            if is_rate_limit:  # changed
                # For rate limits, wait at least 30s (half the 60s window)
                rate_limit_delay = max(30.0, delay)
                print(f"Rate limit hit on attempt {attempt + 1}/{max_retries}. "
                      f"Waiting {rate_limit_delay:.0f}s before retry...")
                time.sleep(rate_limit_delay)
                delay = min(delay * 2, max_delay)
            else:  # changed
                print(f"{error_name} on attempt {attempt + 1}/{max_retries}, "
                      f"waiting {delay:.1f}s: {e}")
                time.sleep(delay)
                delay = min(delay * 2, max_delay)

    raise Exception(f"Failed after {max_retries} attempts: {last_exception}")

---
CHECKPOINTING
---

In [11]:
def create_checkpoint_filepath(output_directory: str, experiment_id: str) -> str:
    """Generate filepath for checkpoint file."""
    return f"{output_directory}/checkpoint_{experiment_id}.json"


def load_checkpoint(checkpoint_filepath: str) -> Optional[ExperimentCheckpoint]:
    """Load existing checkpoint if present."""
    if not Path(checkpoint_filepath).exists():
        return None

    with open(checkpoint_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    return ExperimentCheckpoint(
        experiment_id=data['experiment_id'],
        model_id=data['model_id'],
        split_name=data['split_name'],
        completed_pmcids=set(data['completed_pmcids']),
        failed_pmcids=data['failed_pmcids'],
        last_updated=data['last_updated'],
        total_documents=data['total_documents']
    )


def save_checkpoint(checkpoint: ExperimentCheckpoint, checkpoint_filepath: str) -> None:
    """Save checkpoint to file."""
    checkpoint.last_updated = datetime.now().isoformat()

    data = {
        'experiment_id': checkpoint.experiment_id,
        'model_id': checkpoint.model_id,
        'split_name': checkpoint.split_name,
        'completed_pmcids': list(checkpoint.completed_pmcids),
        'failed_pmcids': checkpoint.failed_pmcids,
        'last_updated': checkpoint.last_updated,
        'total_documents': checkpoint.total_documents
    }

    with open(checkpoint_filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)


def create_new_checkpoint(experiment_id: str,
                          model_id: str,
                          split_name: str,
                          total_documents: int
                          ) -> ExperimentCheckpoint:
    """Create a fresh checkpoint."""
    return ExperimentCheckpoint(
        experiment_id=experiment_id,
        model_id=model_id,
        split_name=split_name,
        completed_pmcids=set(),
        failed_pmcids={},
        last_updated=datetime.now().isoformat(),
        total_documents=total_documents
    )

---
TOKEN TRACKING AND COST CALCULATION
---

In [12]:
def get_actual_token_usage_from_dspy() -> Tuple[int, int, bool]:  # changed - returns 3 values
    """
    Extract actual token usage from DSPy LM history.

    DSPy stores API response metadata in the LM's history attribute.
    This function attempts to retrieve actual token counts from the last API call.

    Returns:
        Tuple of (input_tokens, output_tokens, is_actual)
        - input_tokens: Number of prompt/input tokens
        - output_tokens: Number of completion/output tokens
        - is_actual: True if values are from API, False if unavailable
    """
    try:
        lm = dspy.settings.lm

        if not hasattr(lm, 'history') or not lm.history:
            return 0, 0, False

        last_entry = lm.history[-1]

        # DSPy stores usage in different formats depending on provider
        input_tokens = 0
        output_tokens = 0

        # Pattern 1: Direct usage dict in entry
        if isinstance(last_entry, dict):
            usage = last_entry.get('usage', {})
            if usage:
                input_tokens = usage.get('prompt_tokens', 0) or usage.get('input_tokens', 0)
                output_tokens = usage.get('completion_tokens', 0) or usage.get('output_tokens', 0)

        # Pattern 2: Entry has response with usage
        if input_tokens == 0 and hasattr(last_entry, 'get'):
            response = last_entry.get('response', {})
            if isinstance(response, dict):
                usage = response.get('usage', {})
                input_tokens = usage.get('prompt_tokens', 0) or usage.get('input_tokens', 0)
                output_tokens = usage.get('completion_tokens', 0) or usage.get('output_tokens', 0)

        if input_tokens > 0 or output_tokens > 0:
            return input_tokens, output_tokens, True

        return 0, 0, False

    except Exception as e:
        print(f"Warning: Could not retrieve token usage: {e}")
        return 0, 0, False


def calculate_cost(model_id: str, input_tokens: int, output_tokens: int) -> float:
    """
    Calculate cost in USD for a single inference.
    """
    pricing = MODEL_PRICING.get(model_id, {"input": 0, "output": 0})
    input_cost = (input_tokens / 1_000_000) * pricing["input"]
    output_cost = (output_tokens / 1_000_000) * pricing["output"]
    return input_cost + output_cost

---
REASONING AND HISTORY LOGGING
---

In [13]:
def create_log_filepath(output_directory: str, experiment_id: str) -> str:
    """
    Generate filepath for the reasoning/history log file.
    """
    return f"{output_directory}/{experiment_id}_reasoning_log.txt"


def initialise_log_file(log_filepath: str, experiment_id: str, model_id: str) -> None:
    """
    Create and initialise the reasoning log file with header.
    """
    header = f"""{'=' * 80}
ACCESSION EXTRACTION EXPERIMENT - REASONING AND HISTORY LOG
{'=' * 80}

Experiment ID: {experiment_id}
Model: {model_id}
Started: {datetime.now().isoformat()}
DSPy Version: {dspy.__version__ if hasattr(dspy, '__version__') else 'unknown'}

This log contains:
- Chain-of-Thought reasoning for each document
- Full conversation history from DSPy
- Token usage per call

{'=' * 80}

"""

    with open(log_filepath, 'w', encoding='utf-8') as f:
        f.write(header)


def format_conversation_history() -> str:
    """
    Extract and format conversation history from DSPy LM.
    """
    try:
        lm = dspy.settings.lm

        if not hasattr(lm, 'history') or not lm.history:
            return "[No conversation history available]"

        last_entry = lm.history[-1]

        lines = []

        # Format based on entry structure
        if isinstance(last_entry, dict):
            # Extract messages (prompt)
            messages = last_entry.get('messages', [])
            if messages:
                lines.append("--- MESSAGES (PROMPT) ---")
                for msg in messages:
                    role = msg.get('role', 'unknown')
                    content = msg.get('content', '')
                    # Truncate very long content for readability
                    if len(content) > 2000:
                        content = content[:1000] + "\n[... truncated ...]\n" + content[-500:]
                    lines.append(f"\n[{role.upper()}]")
                    lines.append(content)

            # Extract response
            response = last_entry.get('response', {})
            if response:
                lines.append("\n--- RESPONSE ---")
                if isinstance(response, dict):
                    choices = response.get('choices', [])
                    if choices:
                        for idx, choice in enumerate(choices):
                            message = choice.get('message', {})
                            content = message.get('content', str(choice))
                            lines.append(f"\n[CHOICE {idx}]")
                            lines.append(str(content))
                    else:
                        lines.append(str(response)[:2000])
                else:
                    lines.append(str(response)[:2000])

            # Extract usage
            usage = last_entry.get('usage', {})
            if usage:
                lines.append("\n--- USAGE ---")
                lines.append(json.dumps(usage, indent=2))
        else:
            # Fallback: just stringify the entry
            lines.append(str(last_entry)[:3000])

        return "\n".join(lines)

    except Exception as e:
        return f"[Error formatting conversation history: {e}]"


def log_document_processing(
    log_filepath: str,
    pmcid: str,
    reasoning: str,
    accessions: List[str],
    input_tokens: int,
    output_tokens: int,
    cost_usd: float,
    processing_time_ms: int,
    include_history: bool = True
) -> None:
    """
    Log reasoning and history for a single document processing.
    """
    timestamp = datetime.now().isoformat()

    # Build log entry
    entry_lines = [
        "",
        "=" * 80,
        f"PMCID: {pmcid}",
        f"Timestamp: {timestamp}",
        f"Processing Time: {processing_time_ms} ms",
        "=" * 80,
        "",
        "--- CHAIN-OF-THOUGHT REASONING ---",
        reasoning if reasoning else "[No reasoning captured]",
        "",
        "--- EXTRACTED ACCESSIONS ---",
        f"Count: {len(accessions)}",
        ", ".join(accessions) if accessions else "[None]",
        "",
        "--- TOKEN USAGE ---",
        f"Input tokens: {input_tokens}",
        f"Output tokens: {output_tokens}",
        f"Cost: ${cost_usd:.6f}",
    ]

    # Add conversation history if requested
    if include_history:
        entry_lines.extend([
            "",
            "--- FULL CONVERSATION HISTORY ---",
            format_conversation_history(),
        ])

    entry_lines.append("\n")

    # Append to log file
    with open(log_filepath, 'a', encoding='utf-8') as f:
        f.write("\n".join(entry_lines))


def log_failed_document(
    log_filepath: str,
    pmcid: str,
    error_message: str
) -> None:
    """
    Log a failed document processing.
    """
    timestamp = datetime.now().isoformat()

    entry = f"""
{'=' * 80}
PMCID: {pmcid} [FAILED]
Timestamp: {timestamp}
{'=' * 80}

ERROR: {error_message}

"""

    with open(log_filepath, 'a', encoding='utf-8') as f:
        f.write(entry)


def finalise_log_file(
    log_filepath: str,
    total_documents: int,
    completed: int,
    failed: int,
    total_tokens_input: int,
    total_tokens_output: int,
    total_cost_usd: float
) -> None:
    """
    Add summary footer to log file.
    """
    footer = f"""
{'=' * 80}
EXPERIMENT SUMMARY
{'=' * 80}

Completed: {datetime.now().isoformat()}

Documents:
  Total: {total_documents}
  Completed: {completed}
  Failed: {failed}

Token Usage:
  Input: {total_tokens_input:,}
  Output: {total_tokens_output:,}
  Total: {total_tokens_input + total_tokens_output:,}

Cost: ${total_cost_usd:.4f}

{'=' * 80}
END OF LOG
{'=' * 80}
"""

    with open(log_filepath, 'a', encoding='utf-8') as f:
        f.write(footer)

---
OUTPUT PARSING
---

In [14]:
def parse_model_output(prediction: dspy.Prediction) -> List[str]:
    """
    Parse DSPy prediction to extract accession list.
    """
    accessions_raw = getattr(prediction, 'accessions', [])  # changed - field name

    if accessions_raw is None:
        return []

    if isinstance(accessions_raw, str):
        accessions_raw = re.split(r'[,\n]', accessions_raw)

    accessions = []
    for acc in accessions_raw:
        if acc is None:
            continue
        cleaned = str(acc).strip()
        if cleaned:
            accessions.append(cleaned)

    return accessions


def get_reasoning_from_prediction(prediction: dspy.Prediction) -> str:
    """
    Extract chain-of-thought reasoning from DSPy prediction.

    ChainOfThought stores reasoning in 'reasoning' attribute.
    """
    reasoning = getattr(prediction, 'reasoning', '')

    if reasoning is None:
        reasoning = ''

    return str(reasoning)


def create_extraction_result(pmcid: str,
                             accessions: List[str],
                             raw_output: str,
                             processing_time_ms: int,
                             token_count_input: int,
                             token_count_output: int,
                             tokens_are_estimated: bool,  # changed - added parameter
                             cost_usd: float,
                             model_id: str
                             ) -> ExtractionResult:
    """Create successful extraction result."""
    return ExtractionResult(
        pmcid=pmcid,
        extracted_accessions=accessions,
        raw_model_output=raw_output,
        processing_time_ms=processing_time_ms,
        token_count_input=token_count_input,
        token_count_output=token_count_output,
        tokens_are_estimated=tokens_are_estimated,  # changed
        cost_usd=cost_usd,
        model_id=model_id,
        timestamp=datetime.now().isoformat(),
        status=ExperimentStatus.COMPLETED,
        error_message=None
    )


def create_failed_result(pmcid: str,
                         model_id: str,
                         error_message: str
                         ) -> ExtractionResult:
    """Create failed extraction result."""
    return ExtractionResult(
        pmcid=pmcid,
        extracted_accessions=[],
        raw_model_output="",
        processing_time_ms=0,
        token_count_input=0,
        token_count_output=0,
        tokens_are_estimated=True,  # changed
        cost_usd=0.0,
        model_id=model_id,
        timestamp=datetime.now().isoformat(),
        status=ExperimentStatus.FAILED,
        error_message=error_message
    )

---
MAIN PROCESSING FUNCTIONS
---

In [15]:
def process_single_document(pmcid: str,
                            xml_filepath: str,
                            extractor: AccessionExtractor,
                            format_rules: str,
                            model_id: str,
                            rate_limiter: TokenRateLimiter,  # changed - type updated
                            log_filepath: str
                            ) -> ExtractionResult:
    """
    Process a single document to extract accessions.
    """
    try:
        # Prepare article content
        article = prepare_article_content(pmcid, xml_filepath)

        # Token-based rate limiting - check before making request  # changed
        format_rules_tokens = estimate_token_count(format_rules)
        total_estimated_input = article.token_count_estimate + format_rules_tokens
        wait_for_token_rate_limit(rate_limiter, total_estimated_input)

        # Run extraction with timing
        start_time = time.time()

        def run_extraction():
            return extractor(
                format_rules,
                xml_content=article.extracted_text  # changed - parameter name
            )

        prediction = execute_with_retry(run_extraction)

        end_time = time.time()
        processing_time_ms = int((end_time - start_time) * 1000)

        # Parse output
        accessions = parse_model_output(prediction)

        # Get reasoning from CoT
        reasoning = get_reasoning_from_prediction(prediction)

        # Get raw output for debugging
        raw_output = str(getattr(prediction, 'accessions', ''))  # changed - field name

        # Get actual token counts from DSPy if available  # changed
        actual_input, actual_output, is_actual = get_actual_token_usage_from_dspy()

        if is_actual:  # changed
            token_count_input = actual_input
            token_count_output = actual_output
            tokens_are_estimated = False
            # Record actual tokens for rate limiting
            record_token_usage(rate_limiter, actual_input)
        else:  # changed
            # Fallback to estimates
            token_count_input = total_estimated_input
            token_count_output = estimate_token_count(raw_output)
            tokens_are_estimated = True
            # Record estimated tokens for rate limiting
            record_token_usage(rate_limiter, total_estimated_input)

        # Calculate cost
        cost_usd = calculate_cost(model_id, token_count_input, token_count_output)

        # Log reasoning and history to file
        log_document_processing(
            log_filepath=log_filepath,
            pmcid=pmcid,
            reasoning=reasoning,
            accessions=accessions,
            input_tokens=token_count_input,
            output_tokens=token_count_output,
            cost_usd=cost_usd,
            processing_time_ms=processing_time_ms,
            include_history=True
        )

        # Optionally print reasoning to console
        if PRINT_REASONING:
            print(f"    --- Reasoning ---")
            if len(reasoning) > MAX_REASONING_PRINT_LENGTH:
                print(f"    {reasoning[:MAX_REASONING_PRINT_LENGTH]}...[truncated]")
            else:
                print(f"    {reasoning}")

        # Optionally print history to console
        if PRINT_HISTORY:
            print(f"    --- History ---")
            print(format_conversation_history()[:1000])

        return create_extraction_result(
            pmcid=pmcid,
            accessions=accessions,
            raw_output=raw_output,
            processing_time_ms=processing_time_ms,
            token_count_input=token_count_input,
            token_count_output=token_count_output,
            tokens_are_estimated=tokens_are_estimated,  # changed
            cost_usd=cost_usd,
            model_id=model_id
        )

    except Exception as e:
        # Log failed document
        log_failed_document(log_filepath, pmcid, str(e))

        return create_failed_result(
            pmcid=pmcid,
            model_id=model_id,
            error_message=str(e)
        )


def load_pmcid_to_xml_mapping(xml_directory: str,
                              pmcid_list: List[str]
                              ) -> Dict[str, str]:
    """
    Create mapping of PMCID to XML file paths.
    """
    xml_dir = Path(xml_directory)
    mapping = {}

    pmcid_set = set(pmcid_list)

    for xml_file in xml_dir.glob("*.xml"):
        # Extract PMCID from filename
        filename = xml_file.stem
        pmcid = filename.split('_')[0]

        if pmcid in pmcid_set:
            mapping[pmcid] = str(xml_file)

    return mapping


def load_split_pmcids(ground_truth_filepath: str, split_name: str) -> List[str]:
    """
    Load PMCIDs for a specific split from validation_splits.json.

    Handles both old format (list) and new format (dict with 'files' key).
    """
    with open(ground_truth_filepath, 'r', encoding='utf-8') as f:
        gt_data = json.load(f)

    split_data = gt_data['splits'].get(split_name, {})

    # Handle both old format (list) and new format (dict with 'files' key)
    if isinstance(split_data, dict):
        pmcid_list = split_data.get('files', [])
    else:
        # Old format: split_data is already a list
        pmcid_list = split_data

    return pmcid_list

In [16]:
def run_experiment(model_config: ModelConfig,
                   split_name: str,
                   ground_truth_filepath: str,
                   xml_directory: str,
                   format_rules_filepath: str,
                   output_directory: str,
                   resume_from_checkpoint: bool = True,
                   experiment_id: Optional[str] = None  # changed - added optional parameter
                  ) -> ExperimentOutput:
    """
    Run experiment for a single model.
    """
    # Generate experiment ID
    if experiment_id is None:  # changed
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        experiment_id = f"{model_config.model_id}_{split_name}_{timestamp_str}"

    # Create output directory
    output_dir = Path(output_directory)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load accession format rules
    print(f"Loading format rules from: {format_rules_filepath}")
    format_rules = load_format_rules(format_rules_filepath)

    # Load split pmcids
    pmcid_list = load_split_pmcids(ground_truth_filepath, split_name)
    total_documents = len(pmcid_list)

    print(f"Experiment: {experiment_id}")
    print(f"Model: {model_config.model_id}")
    print(f"Split: {split_name} ({total_documents} documents)")
    print(f"Token rate limit: {model_config.tokens_per_minute:,} tokens/minute")  # changed

    # Initialise log file
    log_filepath = create_log_filepath(output_directory, experiment_id)
    initialise_log_file(log_filepath, experiment_id, model_config.model_id)
    print(f"Reasoning log: {log_filepath}")

    # Load or create checkpoint
    checkpoint_filepath = create_checkpoint_filepath(output_directory, experiment_id)
    checkpoint = None

    if resume_from_checkpoint:
        checkpoint = load_checkpoint(checkpoint_filepath)

    if checkpoint is None:
        checkpoint = create_new_checkpoint(experiment_id, model_config.model_id,
                                           split_name, total_documents)

    # Initialise dspy
    lm = initialise_dspy_model(model_config)
    dspy.configure(lm=lm)

    # Create extractor and token-based rate limiter  # changed
    extractor = create_accession_extractor()
    rate_limiter = create_token_rate_limiter(model_config.tokens_per_minute)  # changed

    # Load pmcid to XML mapping
    pmcid_to_xml = load_pmcid_to_xml_mapping(xml_directory, pmcid_list)

    # Process documents
    predictions = []
    total_processing_time_ms = 0
    total_tokens_input = 0
    total_tokens_output = 0
    total_cost_usd = 0.0

    for idx, pmcid in enumerate(pmcid_list):
        # Skip if completed
        if pmcid in checkpoint.completed_pmcids:
            print(f"  [{idx+1}/{total_documents}] {pmcid} - SKIPPED (already completed)")
            continue

        # Check if XML exists
        if pmcid not in pmcid_to_xml:
            result = create_failed_result(pmcid, model_config.model_id, "XML file not found")
            checkpoint.failed_pmcids[pmcid] = "XML file not found"
            log_failed_document(log_filepath, pmcid, "XML file not found")
        else:
            # Show current rate limit status  # changed
            tokens_used = get_tokens_used_in_window(rate_limiter)
            print(f"  [{idx+1}/{total_documents}] {pmcid} - Processing... "
                  f"(rate: {tokens_used:,}/{model_config.tokens_per_minute:,} tokens/min)")

            result = process_single_document(
                pmcid=pmcid,
                xml_filepath=pmcid_to_xml[pmcid],
                extractor=extractor,
                format_rules=format_rules,
                model_id=model_config.model_id,
                rate_limiter=rate_limiter,
                log_filepath=log_filepath
            )

            if result.status == ExperimentStatus.COMPLETED:
                checkpoint.completed_pmcids.add(pmcid)
                token_info = "actual" if not result.tokens_are_estimated else "est"  # changed
                print(f"    Found {len(result.extracted_accessions)} accessions "
                      f"(tokens: {result.token_count_input:,}/{result.token_count_output:,} [{token_info}], "
                      f"cost: ${result.cost_usd:.4f})")
            else:
                checkpoint.failed_pmcids[pmcid] = result.error_message
                print(f"   FAILED: {result.error_message}")

        predictions.append(result)
        total_processing_time_ms += result.processing_time_ms
        total_tokens_input += result.token_count_input
        total_tokens_output += result.token_count_output
        total_cost_usd += result.cost_usd

        # Save checkpoint periodically
        if (idx + 1) % 10 == 0:
            save_checkpoint(checkpoint, checkpoint_filepath)
            print(f"    [Checkpoint saved. Running totals: {total_tokens_input:,} in / "
                  f"{total_tokens_output:,} out tokens, ${total_cost_usd:.4f}]")

    # Final checkpoint save
    save_checkpoint(checkpoint, checkpoint_filepath)

    # Finalise log file
    finalise_log_file(
        log_filepath=log_filepath,
        total_documents=total_documents,
        completed=len(checkpoint.completed_pmcids),
        failed=len(checkpoint.failed_pmcids),
        total_tokens_input=total_tokens_input,
        total_tokens_output=total_tokens_output,
        total_cost_usd=total_cost_usd
    )

    # Create output
    metadata = ExperimentMetadata(
        experiment_id=experiment_id,
        model_id=model_config.model_id,
        dspy_version=dspy.__version__ if hasattr(dspy, '__version__') else "unknown",
        run_timestamp=datetime.now().isoformat(),
        split_name=split_name,
        total_documents=total_documents,
        completed_documents=len(checkpoint.completed_pmcids),
        failed_documents=len(checkpoint.failed_pmcids),
        total_processing_time_ms=total_processing_time_ms,
        total_tokens_input=total_tokens_input,
        total_tokens_output=total_tokens_output,
        total_cost_usd=total_cost_usd
    )

    output = ExperimentOutput(
        experiment_metadata=metadata,
        predictions=predictions
    )

    # Save output
    output_filepath = output_dir / f"{model_config.model_id}_{split_name}_predictions.json"
    save_experiment_output(output, str(output_filepath))

    print(f"\nExperiment complete: {output_filepath}")
    print(f"  Completed: {metadata.completed_documents}")
    print(f"  Failed: {metadata.failed_documents}")
    print(f"  Total tokens: {total_tokens_input:,} input / {total_tokens_output:,} output")
    print(f"  Total cost: ${total_cost_usd:.4f}")
    print(f"  Reasoning log: {log_filepath}")

    return output


def save_experiment_output(output: ExperimentOutput, output_filepath: str) -> None:
    """Save experiment output to JSON file."""
    predictions_list = []

    for result in output.predictions:
        predictions_list.append({
            "pmcid": result.pmcid,
            "extracted_accessions": result.extracted_accessions,
            "raw_model_output": result.raw_model_output,
            "processing_time_ms": result.processing_time_ms,
            "token_count_input": result.token_count_input,
            "token_count_output": result.token_count_output,
            "tokens_are_estimated": result.tokens_are_estimated,  # changed
            "cost_usd": result.cost_usd,
            "timestamp": result.timestamp,
            "status": result.status.value,
            "error_message": result.error_message
        })

    output_dict = {
        "experiment_metadata": {
            "experiment_id": output.experiment_metadata.experiment_id,
            "model_id": output.experiment_metadata.model_id,
            "dspy_version": output.experiment_metadata.dspy_version,
            "run_timestamp": output.experiment_metadata.run_timestamp,
            "split_name": output.experiment_metadata.split_name,
            "total_documents": output.experiment_metadata.total_documents,
            "completed_documents": output.experiment_metadata.completed_documents,
            "failed_documents": output.experiment_metadata.failed_documents,
            "total_processing_time_ms": output.experiment_metadata.total_processing_time_ms,
            "total_tokens_input": output.experiment_metadata.total_tokens_input,
            "total_tokens_output": output.experiment_metadata.total_tokens_output,
            "total_cost_usd": output.experiment_metadata.total_cost_usd
        },
        "predictions": predictions_list
    }

    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(output_dict, f, indent=2)


def run_all_experiments(model_ids: List[str],
                        split_name: str,
                        ground_truth_filepath: str,
                        xml_directory: str,
                        format_rules_filepath: str,
                        output_directory: str
                        ) -> List[ExperimentOutput]:
    """
    Run experiments for multiple models.
    """
    configs = create_model_configs()
    outputs = []

    for model_id in model_ids:
        if model_id not in configs:
            print(f"Unknown model: {model_id}")
            continue

        config = configs[model_id]

        try:
            output = run_experiment(
                model_config=config,
                split_name=split_name,
                ground_truth_filepath=ground_truth_filepath,
                xml_directory=xml_directory,
                format_rules_filepath=format_rules_filepath,
                output_directory=output_directory,
                resume_from_checkpoint=True
            )
            outputs.append(output)

        except Exception as e:
            print(f"Experiment failed for {model_id}: {e}")

    print(f"\nAll experiments complete. {len(outputs)} successful.")
    return outputs

---
SETUP AND EXECUTION - DEV
---

In [21]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Set path directory
BASE_DIR = Path("/content/drive/MyDrive/AI6129")
OUTPUT_DIR = BASE_DIR / "accession/experiment_outputs"
GROUND_TRUTH_FILE = BASE_DIR / "accession/validation_splits/validation_splits.json"
FORMAT_RULES = BASE_DIR / "accession_formats.md"
XML_FILES_DIR = BASE_DIR / "xml"

print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Ground truth: {GROUND_TRUTH_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base directory: /content/drive/MyDrive/AI6129
Output directory: /content/drive/MyDrive/AI6129/accession/experiment_outputs
Ground truth: /content/drive/MyDrive/AI6129/accession/validation_splits/validation_splits.json


In [22]:
# Load API key from Colab secrets
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('CLAUDE_API_KEY')
    print("Anthropic API key loaded from Colab secrets")
except Exception as e:
    print(f"Warning: Could not load API key from secrets: {e}")
    print("Please set ANTHROPIC_API_KEY manually")

Anthropic API key loaded from Colab secrets


In [23]:
# =============================================================================
# CONFIGURATION - Adjust these before running
# =============================================================================

# Set to True to see reasoning in console output (verbose)
PRINT_REASONING = False

# Set to True to see conversation history in console (very verbose)
PRINT_HISTORY = False

# Maximum reasoning length to print (if PRINT_REASONING is True)
MAX_REASONING_PRINT_LENGTH = 1000

In [24]:
# Run experiment for development set with Haiku 4.5 baseline
output = run_experiment(
    model_config=create_model_configs()["claude-haiku-4.5"],  # changed
    split_name="development",
    ground_truth_filepath=str(GROUND_TRUTH_FILE),
    xml_directory=str(XML_FILES_DIR),
    format_rules_filepath=str(FORMAT_RULES),
    output_directory=str(OUTPUT_DIR),
    resume_from_checkpoint=True,
    experiment_id="claude-haiku-4.5_development"  # changed - explicit ID for consistency
)

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded from: /content/drive/MyDrive/AI6129/accession_formats.md (2334 characters)
Experiment: claude-haiku-4.5_development
Model: claude-haiku-4.5
Split: development (45 documents)
Token rate limit: 40,000 tokens/minute
Reasoning log: /content/drive/MyDrive/AI6129/accession/experiment_outputs/claude-haiku-4.5_development_reasoning_log.txt
  [1/45] PMC9387262 - Processing... (rate: 0/40,000 tokens/min)
    Found 556 accessions (tokens: 51,825/5,385 [actual], cost: $0.0788)
  [2/45] PMC8254292 - Processing... (rate: 51,825/40,000 tokens/min)
Token rate limit: 51,825/40,000 used. Need 16,037 tokens. Waiting 61.0s...
    Found 0 accessions (tokens: 18,833/615 [actual], cost: $0.0219)
  [3/45] PMC7054696 - Processing... (rate: 18,833/40,000 tokens/min)
    Found 0 accessions (tokens: 2,703/484 [actual], cost: $0.0051)
  [4/45] PMC9190773 - Processing... (rate: 21,536/40,000 tokens/min)
    Found 0 acc

---
RUN ON VALIDATION SET
---

In [25]:
# Run experiment for validation set with Haiku 4.5
output_val = run_experiment(
    model_config=create_model_configs()["claude-haiku-4.5"],
    split_name="validation",
    ground_truth_filepath=str(GROUND_TRUTH_FILE),
    xml_directory=str(XML_FILES_DIR),
    format_rules_filepath=str(FORMAT_RULES),
    output_directory=str(OUTPUT_DIR),
    resume_from_checkpoint=True,
    experiment_id="claude-haiku-4.5_validation"
)

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded from: /content/drive/MyDrive/AI6129/accession_formats.md (2334 characters)
Experiment: claude-haiku-4.5_validation
Model: claude-haiku-4.5
Split: validation (117 documents)
Token rate limit: 40,000 tokens/minute
Reasoning log: /content/drive/MyDrive/AI6129/accession/experiment_outputs/claude-haiku-4.5_validation_reasoning_log.txt
  [1/117] PMC8744364 - Processing... (rate: 0/40,000 tokens/min)
    Found 0 accessions (tokens: 5,141/575 [actual], cost: $0.0080)
  [2/117] PMC7460271 - Processing... (rate: 5,141/40,000 tokens/min)
    Found 0 accessions (tokens: 6,497/574 [actual], cost: $0.0094)
  [3/117] PMC7696838 - Processing... (rate: 11,638/40,000 tokens/min)
    Found 0 accessions (tokens: 17,119/654 [actual], cost: $0.0204)
  [4/117] PMC7366320 - Processing... (rate: 28,757/40,000 tokens/min)
Token rate limit: 28,757/40,000 used. Need 15,681 tokens. Waiting 46.0s...
    Found 0 accessi

---
RUN ON TEST SET
---

In [26]:
# Run experiment for test set with Haiku 4.5
output_test = run_experiment(
    model_config=create_model_configs()["claude-haiku-4.5"],
    split_name="test",
    ground_truth_filepath=str(GROUND_TRUTH_FILE),
    xml_directory=str(XML_FILES_DIR),
    format_rules_filepath=str(FORMAT_RULES),
    output_directory=str(OUTPUT_DIR),
    resume_from_checkpoint=True,
    experiment_id="claude-haiku-4.5_test"
)

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded from: /content/drive/MyDrive/AI6129/accession_formats.md (2334 characters)
Experiment: claude-haiku-4.5_test
Model: claude-haiku-4.5
Split: test (65 documents)
Token rate limit: 40,000 tokens/minute
Reasoning log: /content/drive/MyDrive/AI6129/accession/experiment_outputs/claude-haiku-4.5_test_reasoning_log.txt
  [1/65] PMC9347385 - Processing... (rate: 0/40,000 tokens/min)
    Found 0 accessions (tokens: 13,020/0 [est], cost: $0.0130)
  [2/65] PMC4939201 - Processing... (rate: 13,020/40,000 tokens/min)
    Found 0 accessions (tokens: 4,897/0 [est], cost: $0.0049)
  [3/65] PMC7270754 - Processing... (rate: 17,917/40,000 tokens/min)
    Found 0 accessions (tokens: 5,644/0 [est], cost: $0.0056)
  [4/65] PMC7705031 - Processing... (rate: 23,561/40,000 tokens/min)
    Found 0 accessions (tokens: 15,809/0 [est], cost: $0.0158)
  [5/65] PMC5388333 - Processing... (rate: 39,370/40,000 tokens/min)